# Chaos Theory and the Logistic Map
**Category: Mathematics**

## 1. Introduction

The logistic map is one of the simplest dynamical systems to exhibit chaos — deterministic, rule-governed behaviour that is nevertheless practically unpredictable. Introduced in a landmark 1976 paper by Robert May, it became the canonical example demonstrating that complexity can emerge from an elementary one-line equation. Edward Lorenz's observation that a butterfly flapping its wings in Brazil might set off a tornado in Texas is the same phenomenon.

## 2. The Logistic Map

### 2.1 Definition

The logistic map is the discrete-time recurrence
$$
x_{n+1} = r\, x_n (1 - x_n), \qquad x_n \in [0, 1],\; r \in [0, 4]
$$
It models population dynamics where $x_n$ is the population fraction (relative to carrying capacity) and $r$ is the growth rate. For $r > 4$ the interval $[0,1]$ is no longer invariant.

### 2.2 Fixed Points and Stability

Fixed points satisfy $x^* = rx^*(1 - x^*)$, giving $x^* = 0$ and $x^* = 1 - 1/r$ (for $r > 1$). A fixed point $x^*$ is **stable** iff $|f'(x^*)| < 1$, where $f(x) = rx(1-x)$:
$$
f'(x) = r(1 - 2x)
$$
So $x^* = 1 - 1/r$ is stable when $|r(1 - 2(1 - 1/r))| = |2 - r| < 1$, i.e., $1 < r < 3$.

### 2.3 Period Doubling and the Feigenbaum Constant

As $r$ increases past $3$, the fixed point loses stability through a **period-doubling bifurcation** cascade:

| $r$ range | Attractor |
|---|---|
| $0 < r < 1$ | $x^* = 0$ |
| $1 < r < 3$ | $x^* = 1 - 1/r$ |
| $3 < r < 3.449$ | period-2 cycle |
| $3.449 < r < 3.544$ | period-4 cycle |
| $r \to r_\infty \approx 3.5699$ | chaos onset |

The ratio of successive bifurcation intervals converges to the **Feigenbaum constant**
$$
\delta = \lim_{n \to \infty} \frac{r_n - r_{n-1}}{r_{n+1} - r_n} \approx 4.6692
$$
Remarkably, $\delta$ is universal — it appears in any smooth unimodal map, regardless of the specific function.

## 3. The Lyapunov Exponent

### 3.1 Definition and Interpretation

The **Lyapunov exponent** quantifies the rate of divergence of nearby trajectories:
$$
\lambda = \lim_{N \to \infty} \frac{1}{N} \sum_{n=0}^{N-1} \ln |f'(x_n)| = \lim_{N\to\infty} \frac{1}{N} \sum_{n=0}^{N-1} \ln |r(1 - 2x_n)|
$$
- $\lambda < 0$: stable periodic orbit — perturbations shrink
- $\lambda = 0$: bifurcation point
- $\lambda > 0$: chaos — nearby initial conditions separate exponentially

## 4. Numerical Exploration

### 4.1 Bifurcation Diagram and Lyapunov Spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

r_vals = np.linspace(2.5, 4.0, 3000)
n_transient = 500
n_plot = 300

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Bifurcation diagram
for r in r_vals:
    x = 0.5
    for _ in range(n_transient):
        x = r * x * (1 - x)
    xs = []
    for _ in range(n_plot):
        x = r * x * (1 - x)
        xs.append(x)
    axes[0].plot([r] * n_plot, xs, ',', color='black', alpha=0.2, markersize=0.3)

axes[0].set_ylabel('$x_n$ (attractor)')
axes[0].set_title('Bifurcation Diagram of the Logistic Map')

# Lyapunov exponent
lam = []
N = 2000
for r in r_vals:
    x = 0.5
    for _ in range(n_transient):
        x = r * x * (1 - x)
    log_sum = 0
    for _ in range(N):
        x = r * x * (1 - x)
        deriv = abs(r * (1 - 2*x))
        log_sum += np.log(deriv) if deriv > 0 else -20
    lam.append(log_sum / N)

axes[1].plot(r_vals, lam, linewidth=0.8, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1, label='$\\lambda = 0$')
axes[1].set_xlabel('$r$')
axes[1].set_ylabel('Lyapunov exponent $\\lambda$')
axes[1].set_title('Lyapunov Exponent')
axes[1].set_ylim(-3, 1)
axes[1].legend()

plt.tight_layout()
plt.show()

### 4.2 Sensitive Dependence on Initial Conditions

In [ ]:
r = 3.9
x1 = 0.500000
x2 = 0.500001  # differ by 10^-6

N = 80
traj1, traj2 = [x1], [x2]
for _ in range(N - 1):
    x1 = r * x1 * (1 - x1)
    x2 = r * x2 * (1 - x2)
    traj1.append(x1)
    traj2.append(x2)

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
ns = np.arange(N)

axes[0].plot(ns, traj1, linewidth=1.5, label='$x_0 = 0.500000$')
axes[0].plot(ns, traj2, linewidth=1.5, linestyle='--', label='$x_0 = 0.500001$')
axes[0].set_ylabel('$x_n$')
axes[0].set_title(f'Sensitive Dependence on Initial Conditions ($r = {r}$)')
axes[0].legend()

diff = np.abs(np.array(traj1) - np.array(traj2))
axes[1].semilogy(ns, diff + 1e-18)
axes[1].set_xlabel('Iteration $n$')
axes[1].set_ylabel('$|x_n^{(1)} - x_n^{(2)}|$')
axes[1].set_title('Exponential Divergence of Trajectories')

plt.tight_layout()
plt.show()